In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install required packages
!pip install -q transformers diffusers accelerate torchaudio pillow datasets
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torchcodec

print("✓ All packages installed!")

✓ All packages installed!


In [3]:
import os

# ====== UPDATE THESE PATHS ======
# Path to your folder in Google Drive containing the files
DRIVE_FOLDER = "/content/drive/MyDrive/Audio2Image"  # Change this to your folder path

# File names (update if different)
MAIN_PY_FILE = "mlponly.py"
CSV_FILE = "main_dataV1.csv"
ZIP_FOLDER = DRIVE_FOLDER  # Folder containing vggsound_00.zip and vggsound_01.zip

# Verify files exist
print("Checking files...")
print(f"Drive folder: {DRIVE_FOLDER}")
print(f"  Exists: {os.path.exists(DRIVE_FOLDER)}")

main_py_path = os.path.join(DRIVE_FOLDER, MAIN_PY_FILE)
csv_path = os.path.join(DRIVE_FOLDER, CSV_FILE)

print(f"\nFiles:")
print(f"  mlponly.py: {os.path.exists(main_py_path)} - {main_py_path}")
print(f"  CSV: {os.path.exists(csv_path)} - {csv_path}")

# Check for ZIP files
print(f"\nZIP files in {ZIP_FOLDER}:")
for f in os.listdir(ZIP_FOLDER):
    if f.endswith('.zip'):
        print(f"  ✓ {f}")

Checking files...
Drive folder: /content/drive/MyDrive/Audio2Image
  Exists: True

Files:
  mlponly.py: True - /content/drive/MyDrive/Audio2Image/mlponly.py
  CSV: True - /content/drive/MyDrive/Audio2Image/main_dataV1.csv

ZIP files in /content/drive/MyDrive/Audio2Image:
  ✓ vggsound_04.zip
  ✓ vggsound_00.zip
  ✓ vggsound_02.zip


In [4]:
import shutil

# Copy mlponly.py to current directory
if os.path.exists(main_py_path):
    shutil.copy(main_py_path, "/content/mlponly.py")
    print("✓ mlponly.py copied to /content/")
else:
    print(f"❌ ERROR: mlponly.py not found at {main_py_path}")
    print("Please update DRIVE_FOLDER path in the previous cell")

✓ mlponly.py copied to /content/


In [5]:
# Change to content directory
os.chdir("/content")

# Import the training module
import sys
sys.path.insert(0, '/content')

from mlponly import Config, train, infer

print("✓ Training module imported successfully!")

✓ Training module imported successfully!


In [6]:
# Create configuration
cfg = Config()

# Update paths to use Google Drive
cfg.train_csv = csv_path

# **NEW: Enable ZIP support** - No need to extract audio files!
cfg.use_zip_files = True
cfg.audio_folder = DRIVE_FOLDER  # Folder containing vggsound_XX.zip files

# Save checkpoint to Drive (so it persists after Colab session)
cfg.ckpt_path = os.path.join(DRIVE_FOLDER, "audio2image_mapper_dual_mlp_only.pt")

# Training settings (adjust as needed)
cfg.batch_size = 4  # Increase if you have enough GPU memory
cfg.max_epochs = 10  # Start with fewer epochs for testing
cfg.lr = 2e-4

# Multi-task loss weights
cfg.clap_loss_weight = 0.5
cfg.sd_loss_weight = 1.0

# Evaluation settings
cfg.eval_every_n_epochs = 2  # Evaluate every 2 epochs
cfg.num_eval_samples = 2  # Number of samples to generate during eval
cfg.save_eval_images = True

# Print configuration
print("Training Configuration:")
print(f"  Device: {cfg.device}")
print(f"  Batch size: {cfg.batch_size}")
print(f"  Epochs: {cfg.max_epochs}")
print(f"  Learning rate: {cfg.lr}")
print(f"  CSV: {cfg.train_csv}")
print(f"  Checkpoint: {cfg.ckpt_path}")
print(f"  ZIP mode: {cfg.use_zip_files} {'✓ (Direct from ZIP files)' if cfg.use_zip_files else '(From extracted files)'}")

Training Configuration:
  Device: cuda
  Batch size: 4
  Epochs: 10
  Learning rate: 0.0002
  CSV: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
  Checkpoint: /content/drive/MyDrive/Audio2Image/audio2image_mapper_dual_mlp_only.pt
  ZIP mode: True ✓ (Direct from ZIP files)


In [7]:
import torch

print("GPU Information:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current device: {cfg.device}")
else:
    print("  ⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("  Go to Runtime > Change runtime type > Hardware accelerator > GPU")

GPU Information:
  CUDA available: True
  GPU name: NVIDIA A100-SXM4-80GB
  GPU memory: 85.17 GB
  Current device: cuda


In [8]:
from mlponly import AudioCaptionDataset

print("Testing dataset loading...")
try:
    test_ds = AudioCaptionDataset(
        cfg.train_csv,
        audio_folder=cfg.audio_folder,
        use_zip_files=cfg.use_zip_files
    )
    print(f"\n✓ Dataset loaded successfully!")
    print(f"  Total samples: {len(test_ds)}")

    if cfg.use_zip_files:
        print(f"  ZIP files found: {len(test_ds.zip_handles)}")
        for zip_name in test_ds.zip_handles.keys():
            print(f"    - {zip_name}.zip")

    # Test loading first sample
    print("\nTesting first sample...")
    wav, sr, caption = test_ds[0]
    print(f"  Audio shape: {wav.shape}")
    print(f"  Sample rate: {sr}")
    print(f"  Caption: {caption}")
    print("\n✓ Dataset test passed!")

except Exception as e:
    print(f"\n❌ Dataset loading failed: {e}")
    print("\nPlease check:")
    print("  1. CSV file path is correct")
    if cfg.use_zip_files:
        print("  2. ZIP files (vggsound_XX.zip) exist in audio_folder")
        print("  3. CSV format: base_folder,image_file,audio_file,caption")
    else:
        print("  2. Audio files are extracted in the correct location")
        print("  3. CSV format: base_folder,image_file,audio_file,caption")

Testing dataset loading...
Loading dataset from: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
Base directory: /content/drive/MyDrive/Audio2Image
Audio folder: /content/drive/MyDrive/Audio2Image
Use ZIP files: True
Searching for ZIP files...
  ✓ Opened vggsound_04.zip (key: 'vggsound_04', 7523 files)
  ✓ Opened vggsound_00.zip (key: 'vggsound_00', 20003 files)
  ✓ Opened vggsound_02.zip (key: 'vggsound_02', 17334 files)
Skipped header: base_folder,image_file,audio_file,caption
✓ Loaded 18643 audio-caption pairs

✓ Dataset loaded successfully!
  Total samples: 18643
  ZIP files found: 3
    - vggsound_04.zip
    - vggsound_00.zip
    - vggsound_02.zip

Testing first sample...
  Audio shape: torch.Size([482400])
  Sample rate: 48000
  Caption: people marching

✓ Dataset test passed!


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

In [9]:
# Start training
print("Starting training...\n")
train(cfg)
print("\n✅ Training complete!")

Starting training...

Loading dataset from: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
Base directory: /content/drive/MyDrive/Audio2Image
Audio folder: /content/drive/MyDrive/Audio2Image
Use ZIP files: True
Searching for ZIP files...
  ✓ Opened vggsound_04.zip (key: 'vggsound_04', 7523 files)
  ✓ Opened vggsound_00.zip (key: 'vggsound_00', 20003 files)
  ✓ Opened vggsound_02.zip (key: 'vggsound_02', 17334 files)
Skipped header: base_folder,image_file,audio_file,caption
✓ Loaded 18643 audio-caption pairs

Dataset split:
  Training: 16778 samples
  Validation: 1865 samples

Loading CLAP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading SD text encoder for training...
Loading CLIP model for evaluation...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Creating MLP: CLAP audio (512) → CLAP text (512) & SD (768)

Starting Multi-Task Training
Dataset: 18643 samples (16778 train, 1865 val)
Batch size: 4
Epochs: 10
CLAP loss weight: 0.5
SD loss weight: 1.0
Evaluation every: 2 epochs



...

  0%|          | 0/30 [00:00<?, ?it/s]

  Batch 2/3: Avg CLIP = 24.119


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  Batch 3/3: Avg CLIP = 26.400

CLIP Evaluation Results:
  Average CLIP Score: 23.0220
  Evaluated 6 samples from validation set
    Sample 1: 'printer printing...' | CLIP: 19.311
      Saved to: eval_images/epoch10_sample1_score19.31.png
    Sample 2: 'rapping...' | CLIP: 17.782
      Saved to: eval_images/epoch10_sample2_score17.78.png
    Sample 3: 'roller coaster running...' | CLIP: 17.069
      Saved to: eval_images/epoch10_sample3_score17.07.png
    Sample 4: 'female singing...' | CLIP: 31.170
      Saved to: eval_images/epoch10_sample4_score31.17.png

🎯 New best model! CLIP Score: 23.0220
   Saved to: /content/drive/MyDrive/Audio2Image/audio2image_mapper_dual_mlp_only_best.pt

Training completed!
Best CLIP Score: 23.0220

✅ Training complete!
